## _Disclaimer: this notebook is a workflow example only, not a science-ready analysis._

# Asynchronous cutouts with the LIneA service: IC 1816

## 1. What this notebook teaches
The goal is to request multi-band FITS cutouts from the **LIneA Cutout Service** with the asynchronous API and leave with a client you can reuse.

The example target is **IC 1816** (RA 02h 31m 51.0s, Dec −36° 40′ 19.8″), a Seyfert 2 galaxy. The service returns Dark Energy Survey DR2 images in $g, r, i, z, Y$. The surface-brightness plot at the end only checks that those files can be measured.

**Async workflow:**
1. `POST /api/async` with the survey, position, bands, and `PHASE=RUN`. The job lives at the `Location` URL of the redirect.
2. `GET {job}/phase` until the plain-text phase is `COMPLETED`, `ERROR`, or `ABORTED`.
3. `GET {job}`, match each result to its filter by `sequence`, and download the FITS with the same HTTP session.

In [ ]:
# [CELL 1] - IMPORTS & REPRODUCIBILITY METADATA
from datetime import datetime, timezone
from pathlib import Path
import platform
import time
import warnings

import astropy
from astropy.visualization import AsinhStretch, ImageNormalize, ZScaleInterval
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import photutils
from photutils.isophote import Ellipse, EllipseGeometry
import requests
import scipy
from scipy.interpolate import interp1d

from utils import (
    cache_band_cutout,
    ensure_cache_dir,
    extract_forced_isophotes,
    prepare_band_image,
)

plt.rcParams.update({"font.size": 11, "axes.linewidth": 1.2})

print(f"Execution Timestamp (UTC) : {datetime.now(timezone.utc).isoformat()}")
print(f"Python                    : {platform.python_version()}")
print(f"NumPy / SciPy             : {np.__version__} / {scipy.__version__}")
print(f"Astropy / Photutils       : {astropy.__version__} / {photutils.__version__}")
print(f"Matplotlib / Requests     : {matplotlib.__version__} / {requests.__version__}")

## 2. Submit the job

Images are **Dark Energy Survey Data Release 2** cutouts (Abbott et al. 2021), served by LIneA (*Laboratório Interinstitucional de e-Astronomia*). DES DR2 does not need a token. One `requests.Session` has to cover the whole job: the `POST` sets a cookie that the later downloads require.

The body uses the SODA cutout parameters:

| Field | Role |
| --- | --- |
| `ID` | Survey identifier. Here, `des_dr2`. |
| `POS` | `CIRCLE <ra> <dec> <radius>` with ra, dec, and radius in degrees. |
| `BAND` | One value per filter. The list below is sent as repeated fields. |
| `format` | `fits`. |
| `PHASE` | `RUN` starts the job immediately. |

`allow_redirects=False` is required. A successful submit answers HTTP 303, and the `Location` header is the job URL (`/api/async/{job_id}`). Every later call uses that URL.

`radius_arcmin = 2` keeps the demo small. Divide by 60 before putting the radius in `POS`. Copies on the server expire after 7 days, so each FITS is cached under `cutouts/`. Delete that folder to force a new request.

In [ ]:
# Submit one async cutout job for IC 1816.
# J2000: 02h31m51.0s, -36d40m19.8s.
ra, dec = 37.962438, -36.672086
radius_arcmin = 2.0
radius_deg = radius_arcmin / 60.0  # SODA CIRCLE radius is in degrees

bands = ["g", "r", "i", "z", "Y"]
colors_dict = {"g": "#1f77b4", "r": "#2ca02c", "i": "#d62728", "z": "#9467bd", "Y": "#8c564b"}
cache_dir = ensure_cache_dir("cutouts")

# Reuse the same Session to preserve cookies for downloads (DES DR2 does not require a token)
session = requests.Session()
# import os
# if "CUTOUT_TOKEN" in os.environ:
#     session.headers["Authorization"] = f"Token {os.environ['CUTOUT_TOKEN']}"

pos = f"CIRCLE {ra} {dec} {radius_deg:.6f}"
response = session.post(
    "https://cutout.linea.org.br/api/async",
    data={
        "ID": "des_dr2",
        "POS": pos,
        "BAND": bands,
        "format": "fits",
        "PHASE": "RUN",
    },
    allow_redirects=False,
    timeout=60,
)
response.raise_for_status()

job_url = response.headers["Location"]
print(f"POS = {pos}")
print(f"HTTP {response.status_code}; job URL: {job_url}")
print("Waiting for the async job (often seconds, sometimes a few minutes)...")

# Poll phase + per-band task status until the job ends.
phase = ""
seen = None
detail = {}
t0 = time.monotonic()
for _ in range(120):
    phase = session.get(f"{job_url}/phase", timeout=60).text.strip()
    detail = session.get(job_url, timeout=60).json()
    tasks_now = sorted(detail.get("tasks", []), key=lambda item: item.get("sequence") or 0)
    band_status = ", ".join(
        f"{task.get('band', '?')}={task.get('status', '?')}" for task in tasks_now
    ) or "no tasks yet"
    snapshot = f"{phase}|{band_status}"
    elapsed = time.monotonic() - t0
    if snapshot != seen:
        print(f"[{elapsed:5.1f}s] phase: {phase} | {band_status}")
        seen = snapshot
    if phase in {"COMPLETED", "ERROR", "ABORTED"}:
        break
    time.sleep(3)
else:
    raise TimeoutError(f"Cutout job timed out after {time.monotonic() - t0:.1f}s in phase: {phase}")

if phase != "COMPLETED":
    errors = [
        f"Task {i} ({t.get('band', '?')}): {t.get('error_message')}"
        for i, t in enumerate(detail.get("tasks", []))
        if t.get("error_message")
    ]
    raise RuntimeError(f"Job ended in {phase}: {'; '.join(errors) or detail}")

print(f"Job COMPLETED in {time.monotonic() - t0:.1f}s")

## 3. Map each file from the completed job

The submit cell already polled `GET {job_url}/phase` (plain text) and printed per-band task status until `COMPLETED`. A finished job document has two lists:

- `results` has one file per band, with `sequence`, `result_id`, and `download_url`. The id contains the filter, as in `..._astrocut_g_1`.
- `tasks` also has `band` and `sequence`, but the task list is not in result order. Match the two by `sequence`. Indexing `tasks[i]` assigns the wrong filter.

Download each `download_url` with the same session. Before trusting the file name, read the FITS keyword `BAND`. `cache_band_cutout` does that match and replaces a cached file whose header disagrees.

In [ ]:
# tasks[] is not ordered like results[]. Pair the two lists by sequence.
tasks = detail.get("tasks", [])
print("tasks:")
for task in sorted(tasks, key=lambda item: item.get("sequence") or 0):
    print(f"  sequence {task.get('sequence')}: band {task.get('band')} ({task.get('status')})")

fits_paths = {}
for res in detail.get("results", []):
    band, file_path = cache_band_cutout(session, res, tasks, bands, cache_dir, radius_arcmin)
    if band in fits_paths:
        warnings.warn(f"Band {band} was assigned to more than one result; check the band mapping.")
    fits_paths[band] = file_path
    print(f"Band {band}: saved to {file_path} ({file_path.stat().st_size / 1024**2:.2f} MB)")

missing = [b for b in bands if b not in fits_paths]
if missing:
    warnings.warn(f"No cutout was mapped for bands: {missing}")

## 4. Example: open the cutouts

The cells below only consume the cached FITS. They read `MAGZERO` and the pixel scale from each header, then build a surface-brightness profile so you can see that every band arrived.

Ellipses are fitted on the $r$ image and reused on the other bands, which keeps a color comparison at the same radius. The masking and that forced sampling live in `utils.py`.

In [ ]:
# Example only: calibrate and sample the files the cutout job just cached.

images = {}
calib = {}
for b in bands:
    if b in fits_paths:
        img, zp, ps, sky_err = prepare_band_image(fits_paths[b])
        images[b] = img
        calib[b] = {"zp": zp, "pix_scale": ps, "sky_err": sky_err}
        print(f"Band {b}: MAGZERO = {zp:.2f}, pixel_scale = {ps:.4f} arcsec/px, sky_err = {sky_err:.4f} ADU/s")

# Fit reference geometry on r-band (starting at minsma=6.0 px ~ 1.6 arcsec)
ref_band = "r" if "r" in images else next(iter(images))
ref_img = images[ref_band]
y0, x0 = np.array(ref_img.shape) / 2.0
init_geom = EllipseGeometry(x0=x0, y0=y0, sma=10.0, eps=0.15, pa=np.pi / 4)

ref_ellipse = Ellipse(ref_img, geometry=init_geom)
ref_isolist = ref_ellipse.fit_image(sma0=10.0, minsma=6.0, maxsma=220.0, sclip=3.0, n_clip=2)
if len(ref_isolist) == 0:
    print(f"[Warning] Free fit failed on {ref_band}-band; falling back to fixed geometry.")
    ref_isolist = ref_ellipse.fit_image(
        sma0=10.0, minsma=6.0, maxsma=220.0, fix_center=True, fix_pa=True, fix_eps=True
    )

profiles = {}
for b in bands:
    if b not in images:
        continue
    isolist = ref_isolist if b == ref_band else extract_forced_isophotes(images[b], ref_isolist)
    if len(isolist) == 0:
        continue

    ps = calib[b]["pix_scale"]
    zp = calib[b]["zp"]
    sky_err = calib[b]["sky_err"]

    intens = isolist.intens
    intens_err = np.hypot(np.nan_to_num(isolist.int_err, nan=0.0), sky_err)
    valid = np.isfinite(intens) & (intens > 0)
    if not np.any(valid):
        continue

    r_arcsec = isolist.sma[valid] * ps
    flux_arcsec2 = intens[valid] / (ps**2)
    mu = -2.5 * np.log10(flux_arcsec2) + zp
    mu_err = (2.5 / np.log(10.0)) * (intens_err[valid] / intens[valid])

    profiles[b] = (r_arcsec, mu, mu_err)
    print(f"Band {b}: extracted {len(r_arcsec)} valid isophotes (r = {r_arcsec.min():.1f}'' to {r_arcsec.max():.1f}'').")

Colors are $g - X = \mu_g - \mu_X$, with band $X$ interpolated onto the $g$ radii. The dashed line at $2.2''$ is about twice the typical DES seeing: inside it, the filters are not blurred to a common PSF.

In [ ]:
# [CELL 5] - RADIAL COLOR GRADIENTS & ERROR PROPAGATION
color_profiles = {}

if "g" in profiles and len(profiles["g"][0]) > 1:
    r_g, mu_g, err_g = profiles["g"]
    for target in ["r", "i", "z"]:
        if target in profiles and len(profiles[target][0]) > 1:
            r_t, mu_t, err_t = profiles[target]
            mu_t_i = interp1d(r_t, mu_t, kind="linear", bounds_error=False, fill_value=np.nan)(r_g)
            err_t_i = interp1d(r_t, err_t, kind="linear", bounds_error=False, fill_value=np.nan)(r_g)

            color_val = mu_g - mu_t_i
            color_err = np.hypot(err_g, err_t_i)
            valid = ~np.isnan(color_val)

            if np.any(valid):
                color_profiles[f"g-{target}"] = (r_g[valid], color_val[valid], color_err[valid])
                print(f"Color profile (g - {target}) computed ({np.sum(valid)} radial points).")
else:
    print("[Error] Valid g-band profile required to compute color gradients.")

In [ ]:
# [CELL 6] - DIAGNOSTIC PLOTS: STAMPS, SURFACE BRIGHTNESS, AND COLOR GRADIENTS
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 10, height_ratios=[1, 1.5], hspace=0.32)

# Row 1: 5 FITS Cutout Stamps evenly spaced across 10 columns
for idx, band in enumerate(bands):
    ax = fig.add_subplot(gs[0, 2 * idx : 2 * idx + 2])
    if band in images:
        raw_view = images[band].data
        norm = ImageNormalize(raw_view, interval=ZScaleInterval(), stretch=AsinhStretch())
        ax.imshow(raw_view, origin="lower", cmap="gray", norm=norm)
        ax.set_title(f"{band}-band", fontsize=13, fontweight="bold")
    ax.set_xticks([])
    ax.set_yticks([])

# Row 2 (Left): Surface Brightness Profiles with 1-sigma error bars
ax_mu = fig.add_subplot(gs[1, :5])
for band in bands:
    if band in profiles:
        r_p, mu_p, err_p = profiles[band]
        ax_mu.errorbar(
            r_p, mu_p, yerr=err_p, fmt="o-", color=colors_dict[band],
            markersize=4, capsize=2, alpha=0.85, label=f"{band}-band"
        )

ax_mu.invert_yaxis()
ax_mu.set_title("Forced-Photometry Surface Brightness Profile", fontsize=14, pad=10)
ax_mu.set_xlabel("Semi-major Axis $r$ (arcsec)", fontsize=12)
ax_mu.set_ylabel(r"$\mu \quad (\mathrm{mag\,arcsec^{-2}})$", fontsize=12)
ax_mu.grid(True, linestyle=":", alpha=0.7)
ax_mu.legend(fontsize=11, loc="upper right")

# Row 2 (Right): Radial Color Profiles with propagated uncertainties
ax_col = fig.add_subplot(gs[1, 5:])
color_scheme = {"g-r": "#2ca02c", "g-i": "#d62728", "g-z": "#9467bd"}

for label, (r_c, val_c, err_c) in color_profiles.items():
    ax_col.errorbar(
        r_c, val_c, yerr=err_c, fmt="o-", color=color_scheme.get(label, "black"),
        markersize=4, capsize=2, alpha=0.85, label=f"({label})"
    )

ax_col.axvline(2.2, color="gray", linestyle="--", alpha=0.6, label=r"2x FWHM ($\sim 2.2''$)")
ax_col.set_title("Radial Color Gradients (Fixed $r$-band Geometry)", fontsize=14, pad=10)
ax_col.set_xlabel("Semi-major Axis $r$ (arcsec)", fontsize=12)
ax_col.set_ylabel("Color Index (mag)", fontsize=12)
ax_col.grid(True, linestyle=":", alpha=0.7)
ax_col.legend(fontsize=11, loc="upper right")

plt.show()